# Module 4 — Add AgentCore Observability

Your agent is **built** (Module 1) and **deployed** (Module 2). The last rung makes it **observable**:
you'll *see* exactly what every request does — which tools it called, how many tokens it used, how long
each step took, and where it failed — in the **Amazon CloudWatch GenAI Observability** dashboard.

### What it takes (no agent code changes)

Observability adds **zero agent code** — the agent here is byte-identical to Module 2. It's three
operational switches plus a look at the dashboard:

| Step | What happens | Where |
|------|--------------|-------|
| **1. Transaction Search** | Account-level switch — makes spans searchable in `/aws/spans` | one-time, account (a script) |
| **2. Deploy** | The agent's image already runs under `opentelemetry-instrument` (ADOT), so it **emits** OTEL spans | `agentcore deploy` |
| **3. Runtime Tracing toggle** | Per-runtime switch — **delivers** the agent's spans to CloudWatch | console, per deployed agent |
| **4. Invoke + view** | Generate traffic (with a session id) and read the trace waterfall | console dashboard |

> Two distinct things have to be true: the agent must **emit** spans (Step 2 — the container's
> `opentelemetry-instrument` wrapper) *and* the runtime must be told to **deliver** them (Step 3 — the
> Tracing toggle). Account-level Transaction Search (Step 1) makes them searchable.

## Why observability?

When an agent runs autonomously through a multi-step loop, "the answer looks wrong" tells you almost
nothing. Observability turns the black box into a glass box:

- **Trace waterfall** — every step and tool call, for debugging
- **Token & cost** — what each invocation costs
- **Session correlation** — group everything by user/session for support
- **Latency & errors** — find the slow step, see where it failed

A production agent you can't see into is one you can't trust or operate.

## What you'll see in a trace (reading, not writing)

The runtime emits spans following **OpenTelemetry GenAI semantic conventions**, which is what lets the
CloudWatch dashboard render them as an agent trace:

```
invoke_agent cos                         ← top-level span for one request
├── gen_ai.operation.name = "invoke_agent"
├── gen_ai.usage.input_tokens / output_tokens
├── session.id = "<your session id>"     ← groups invocations in the same conversation
└── execute_tool <name>                  ← one child span per tool the agent used
    ├── Bash  (ran a script)
    ├── Read  (read financial_data / CLAUDE.md)
    └── Task  (delegated to a subagent)
```

You **read** these conventions to interpret a trace. You do **not** hand-write spans — the AgentCore
runtime does the instrumentation for you. (That's the modern, recommended path; older examples that
hand-built GenAI spans are no longer necessary for runtime-hosted agents.)

## Setup

Run the cell below to install all dependencies and register the Jupyter kernel.
After it completes, **select the `module-4-observability` kernel** from the kernel picker (top-right)
and continue with the rest of the notebook.

## Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [ ]:
!bash setup.sh

### Setup step 2

once you see the depencies and kernel spec module-xxx are installed per message from the last step, please refresh your brower (not refresh kernel but brower)

![](images/refresh-browser.png)

and once refresh, click on the button (it probably shows a python verion 3.11.15) you used to select kernel in the preview section

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once click, you will see the option of select jupyter kernel and please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once click, you can see our registered module-x kernel, and the example shows modul-1-x but ***please select accordingly depedning on which model you are working on, if you are in module 4, then select module-4-xx***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see the following as your kernel , ***please select accordingly depedning on which model you are working on, if it is other module 4, select module-4-xx kernel***

![](images/example-module-1-jupyter-kernel-selected.png)


### Generate deployment target (`aws-targets.json`)

In [ ]:
import json, os, subprocess

# --- Resolve region: single source of truth for the whole notebook ---
# Priority: AWS_REGION env (set by Workshop Studio) → AWS CLI config → fallback
_cli_region = subprocess.run(
    ["aws", "configure", "get", "region"], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get("AWS_REGION") or _cli_region or "us-west-2"

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True,
).stdout.strip()

# Generate aws-targets.json from the resolved values
targets = [
    {
        "name": "default",
        "description": "Workshop deployment target (auto-generated).",
        "account": account_id,
        "region": REGION,
    }
]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)

print(f"✅ agentcore/aws-targets.json written:")
print(f"   account: {account_id}")
print(f"   region:  {REGION}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import boto3

print(f"Deploy region: {REGION}")

# Confirm AWS identity
try:
    who = boto3.client("sts").get_caller_identity()
    print(f"✅ AWS identity: {who['Arn']}")
except Exception as e:
    print(f"⚠️  AWS credentials not usable: {e}")

## Step 1 — Enable CloudWatch Transaction Search (one-time, account-level)

This is the **only** genuinely new setup in Module 4. Transaction Search is what makes the agent's
OpenTelemetry spans searchable in CloudWatch (they land in the `/aws/spans` log group). It's an
account-level switch — you do it once, not per deployment.

The helper is **idempotent** (safe to re-run): it checks the current state and only changes what's
missing.

In [ ]:
result = subprocess.run(
    ["python", "scripts/enable_transaction_search.py", "--region", REGION],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
# Note: after first enabling, allow ~10 minutes before spans are fully searchable.

## Step 2 — Deploy the (already-observable) agent

This is the *same* deploy as Module 2 — nothing extra. Because `enableOtel: true` and ADOT are already in
the config/image, the deployed agent is instrumented automatically. Look at the runtime config:

In [ ]:
cfg = json.load(open("agentcore/agentcore.json"))
rt = cfg["runtimes"][0]
print(json.dumps({
    "name": rt["name"], "build": rt["build"], "protocol": rt["protocol"],
    "instrumentation": rt.get("instrumentation"),
}, indent=2))
# enableOtel: true  → the AgentCore runtime wraps the agent with opentelemetry-instrument on deploy.

Deploy from a **terminal** (builds the image in the cloud via CodeBuild, ~several minutes):

```bash
agentcore deploy -y
agentcore status        # confirm the runtime is READY — note its agent id / ARN for the next step
```

The container's `CMD` runs the agent under **`opentelemetry-instrument`** (from `aws-opentelemetry-distro`),
so once deployed the agent **emits** OTEL spans automatically. Next we tell the runtime to **deliver** them.

In [ ]:
!agentcore deploy -y

In [ ]:
!agentcore status

In [ ]:
#check the current region
!echo $REGION

## Step 3 — Generate traffic (with a session id)

Invoke the deployed agent a couple of times, passing a **session id**. The session id is what groups
related invocations together in the dashboard's *Sessions* view.

```bash
agentcore invoke --session-id "m4-observability-demo-session-001" "What is our current runway and cash position?"
agentcore invoke --session-id "m4-observability-demo-session-001" "If we hire 10 engineers, how does that change?"
```

In [ ]:
!agentcore invoke --session-id "m4-observability-demo-session-001" "What is our current runway and cash position?"

In [ ]:
!agentcore invoke --session-id "m4-observability-demo-session-001" "If we hire 10 engineers, how does that change?"

## Step 4 — View the traces

**In the console (the main event):** open the GenAI Observability dashboard — it has **Agents**,
**Sessions**, and **Traces** views. Pick your agent, drill into a session, and open a trace to see the
span waterfall (tool calls, token usage, latency).

```
https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/sessions
```

(Replace `{REGION}`. Allow ~2–10 minutes after invoking for spans to be indexed.)


In [ ]:
import time

print(f"GenAI dashboard: "
      f"https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/sessions")

# Peek at /aws/spans for spans from our session (best-effort; indexing can lag a few minutes).
logs = boto3.client("logs", region_name=REGION)

def find_spans(session_id, attempts=10, delay=30):
    query = (
        "fields @timestamp, @message "
        f"| filter @message like '{session_id}' "
        "| sort @timestamp desc | limit 20"
    )
    for i in range(attempts):
        try:
            start = logs.start_query(
                logGroupName="aws/spans",
                startTime=int(time.time()) - 3600,
                endTime=int(time.time()),
                queryString=query,
            )["queryId"]
        except Exception as e:
            print(f"  ⚠️  Could not query /aws/spans: {e}")
            print("  → Check the console dashboard directly (link above).")
            return False
        # poll this query
        while True:
            r = logs.get_query_results(queryId=start)
            if r["status"] in ("Complete", "Failed", "Cancelled"):
                break
            time.sleep(2)
        rows = r.get("results", [])
        if rows:
            print(f"✅ Found {len(rows)} span(s) for session '{session_id}':")
            for row in rows[:5]:
                d = {f["field"]: f["value"] for f in row}
                print(f"  • {d.get('@timestamp', '?')}")
            return True
        print(f"  …no spans yet (attempt {i+1}/{attempts}); waiting {delay}s for indexing")
        time.sleep(delay)
    print("⚠️  No spans found after polling — check the console dashboard (link above).")
    print("   (Spans can take up to ~10 min to become searchable in Logs Insights.)")
    return False

find_spans(SESSION_ID)

## Step 5 — Reading a trace: a guided walkthrough

Now that spans are flowing, let's learn how to **read** them. This is the skill that turns
observability from "a dashboard exists" into "I can debug my agent in production."


***The number of span and traces may vary, depending on your invocations during the workshop, so the below screenshots are just to show examples***

---

### Level 1: The Session view

We queried for session `m4-observability-demo-session-001` and the dashboard shows this:

![](images/session-001-10-spans.png)

Here's what each column tells you:

| Column | What it means | Why you care |
|--------|---------------|--------------|
| **Trace ID** | A unique identifier for ONE end-to-end invocation (one question → one answer) | Click it to drill into the full span tree |
| **Spans** | How many spans (units of work) this invocation produced | More spans = more tool calls / steps the agent took |
| **Input** | The user's message (truncated) | Quick scan — which request is which? |
| **Output** | The agent's response (truncated) | Verify the agent answered correctly without opening the full trace |
| **Errors** | Count of spans that errored | **0 = healthy.** Non-zero → drill in immediately |
| **Throttles** | Count of throttled API calls | Indicates you're hitting service limits |
| **Avg. span latency** | Mean duration across all spans in this trace | Your rough "response time" signal |
| **Start time** | When the invocation began | Correlate with user-reported "it was slow at 3pm" |

**What we see here:** two traces (our two `agentcore invoke` calls), each with **5 spans**, zero
errors, and latencies of ~5.2s and ~4.0s. The session id groups them together — this is how you'd
find "all of Alice's requests today" in production.

---

### Level 2: Inside a single trace (the span waterfall)

Click a Trace ID and you get the **span tree** — the full breakdown of what happened inside that
one invocation:

![](images/session-001-trace-1.png)

Reading left to right:

**Left panel — the span waterfall (timeline):**

```
POST /invocations                          ← the HTTP request hitting the runtime
└── ClaudeAgentSDK.ClaudeSDKClient...      ← the agent's think-act loop
    ├── Bash (0)                           ← tool call: ran a shell command
    ├── Read (3)                           ← tool calls: read 3 files (financial data, CLAUDE.md, etc.)
    └── ...                                ← any other tool calls
```

Each bar's **width** is proportional to its duration — so you can visually spot which step was slow.
The hierarchy shows **causality**: the `POST /invocations` span is the parent (the whole request),
and everything nested inside it is work the agent did to answer.

**Right panel — span details (when you click a span):**

- **Span name:** e.g. `ClaudeAgentSDK.ClaudeSDKClient.receive_response` — tells you *what kind*
  of operation this was (receiving the model's response)
- **Message content:** the actual input/output text for that span — lets you see exactly what the
  model said or what a tool returned
- **Attributes:** metadata like `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`,
  `session.id`, error codes, etc.


**You will observe the this dashboard shows token 0 and this is expected and known limitation as claude agent sdk uses openinfernce standards to report those metrics (llm.token_count.*) while the claudwatch is expecting OTel GenAI format (gen_ai.usage.*)**
But you can find those token information from the Bedrock model invoation dashboard 

![](images/bedrock-model-invocations.png)


---

### Level 3: What to look for in practice

When you're debugging a real agent in production, here's the mental checklist:

| Symptom | Where to look in the trace |
|---------|---------------------------|
| "The agent is slow" | Find the widest bar in the waterfall — that's your bottleneck (usually a tool call or model inference) |
| "The answer is wrong" | Click the `receive_response` span — read what the model actually generated vs. what context it had |
| "A tool call failed" | Look for spans with error status — click to see the error message and stack trace |
| "It's using too many tokens" | Check `gen_ai.usage.input_tokens` / `output_tokens` on the model spans |
| "It called tools I didn't expect" | Count the child spans — each tool call is one span. More than expected = the agent is looping |

---

### The key mental model

Think of a trace as a **receipt** for one agent invocation:

- **Session** groups receipts by conversation (same user, same thread)
- **Trace** is one receipt (one question → one answer)
- **Span** is one line item on the receipt (one tool call, one model inference, one HTTP hop)

You don't write any of this instrumentation code — the AgentCore runtime + `opentelemetry-instrument`
produces it automatically. Your job is to **read** the receipt when something goes wrong.

## Key takeaways

- AgentCore Runtime **auto-instruments** your agent with OpenTelemetry — observability needs **no agent
  code change** (it was already on from Module 2's `enableOtel` + ADOT).
- The one new step is **enabling CloudWatch Transaction Search** (account-level, one-time).
- Traces follow **GenAI semantic conventions** (`gen_ai.*`, `session.id`), which is why the **GenAI
  Observability dashboard** can render the agent's trace waterfall, tokens, and tool calls.
- Pass a **session id** on invoke to correlate a conversation in the dashboard.

> **Cleanup (optional):** the runtime is serverless — it costs essentially nothing while idle, so we leave it deployed and you can come back and invoke it again anytime. When you *do* want to remove it, run `agentcore remove agent --name cos` then `agentcore deploy -y` in a terminal. (**Transaction Search stays enabled** either way — it's an account-level, one-time setting with no charge at low/no indexing sampling.)

🎉 You've climbed the whole ladder: **built → deployed → memory -> observable.**